# §4 Meaning: the tree-walk evaluator

*Reads `calc/evaluator.py`. Refines the fragment ⟨Walk the tree to a number⟩ from §1.*

The last stage gives the tree a value. Because the parser (§3) already baked precedence into the tree's shape, evaluation has no precedence logic at all — it is a plain post-order walk: evaluate the children, then combine them.

## §4.1 One function, three cases

`evaluate` dispatches on the node kind, mirroring the three node types from §3.1:

📄 [calc/evaluator.py lines 11–27](../sample_repo/calc/evaluator.py)

```python
# calc/evaluator.py : lines 11-27
def evaluate(node: Node) -> float:
    if isinstance(node, Num):
        return node.value
    if isinstance(node, Unary):
        return -evaluate(node.operand)
    if isinstance(node, BinOp):
        left = evaluate(node.left)
        right = evaluate(node.right)
        if node.op == "+":
            return left + right
        if node.op == "-":
            return left - right
        if node.op == "*":
            return left * right
        if node.op == "/":
            return left / right
    raise TypeError(f"cannot evaluate node {node!r}")
```

- A `Num` is its own value — the base case that ends the recursion.
- A `Unary` evaluates its operand and negates it.
- A `BinOp` evaluates **both children first** (this is the post-order), then applies the operator. Because the `*` node sits below the `+` node in `1 + 2 * 3`, the multiplication is evaluated during the recursion into the children, before the addition combines them — precedence, for free, from the tree's shape.

The closing `raise TypeError` is unreachable for trees the parser builds; it documents the invariant ('every node is one of the three kinds') and would catch a future fourth node kind that forgot to update this function.

## See it run

Evaluate a tree, and re-run the whole pipeline for good measure:

In [1]:
import os, sys
# Find the bundled demo package (works when run from the calc-literate directory).
d = os.getcwd()
for _ in range(6):
    for cand in (os.path.join(d, 'sample_repo'), os.path.join(d, '..', 'sample_repo')):
        if os.path.isdir(os.path.join(cand, 'calc')):
            sys.path.insert(0, os.path.abspath(cand))
            break
    else:
        d = os.path.dirname(d); continue
    break
import calc
print('imported calc', calc.__version__)

imported calc 0.1.0


In [2]:
from calc.tokens import tokenize
from calc.parser import parse
from calc.evaluator import evaluate
tree = parse(tokenize('1 + 2 * 3'))
print('tree :', tree)
print('value:', evaluate(tree))
print('assoc:', evaluate(parse(tokenize('8 - 2 - 1'))), '(left-associative, so 5.0)')

tree : BinOp(op='+', left=Num(value=1.0), right=BinOp(op='*', left=Num(value=2.0), right=Num(value=3.0)))
value: 7.0
assoc: 5.0 (left-associative, so 5.0)


## The whole book in one line

We have now read every stage. Composed, they are exactly the `calc()` of §1 — string to tokens to tree to number:

In [3]:
print(evaluate(parse(tokenize('2 * (3 + 4) - 1'))))   # the three stages, by hand
import calc
print(calc.calc('2 * (3 + 4) - 1'))                  # the same thing, via the public API

13.0
13.0


## Closing

`calc` is four ideas: words (§2), structure (§3), meaning (§4), and a thin shell that wires them together (§1). The design decision that organizes everything is making the parser encode precedence in the tree's *shape*, which let the tokenizer stay ignorant of grammar and the evaluator stay ignorant of precedence. Read in dependency order, the program explains itself — which is the whole promise of a literate reading.